## Cardiac Structure Segmentation — 5-Fold Cross-Validation Training

Trains a U-Net (ResNet34 + SCSE + Deep Supervision) on the CAMUS dataset using 5-fold patient-level cross-validation following nnU-Net conventions.

## Setup

This notebook is designed for **Google Colab** with data stored in Google Drive.

**Before running**, update these paths in Cell 0:
- `zip_path`: path to your CAMUS `database_nifti.zip`
- `SAVE_DIR`: path where checkpoints and results will be saved

The CAMUS dataset can be downloaded from: https://www.creatis.insa-lyon.fr/Challenge/camus/

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 0: Environment Setup — Install, Mount Drive, Extract Data
# ══════════════════════════════════════════════════════════════

# Install dependencies
!pip install segmentation-models-pytorch ttach albumentations pytorch-lightning -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Extract CAMUS dataset
import os, zipfile

zip_path = "/content/drive/MyDrive/CAMUS/database_nifti.zip"

print("Unzipping...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("/content")
print("Done!")

# Verify
patients = [f for f in os.listdir("/content/database_nifti") if f.startswith("patient")]
print(f"Number of patients: {len(patients)}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1: Installs, Imports, Constants
# ══════════════════════════════════════════════════════════════


import os
import glob
import random
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

import segmentation_models_pytorch as smp
import ttach as tta
import albumentations as A
import cv2
from scipy import ndimage

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Performance ──
torch.set_float32_matmul_precision('high')

# ── Constants ──
DATA_DIR = '/content/database_nifti'
SAVE_DIR = '/content/drive/MyDrive/CAMUS/results/final'
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE = 256
NUM_CLASSES = 4
BATCH_SIZE = 4
NUM_WORKERS = 4
LR = 3e-4

# ═══ NEW: nnU-Net-style training config ═══
EPOCHS = 1000               # nnU-Net default (was 100)
BATCHES_PER_EPOCH = 250      # nnU-Net: 250 batches = 1 epoch (not full dataset pass)
PATIENCE = 100               # Higher because short epochs (~50% of a full pass each)
N_FOLDS = 5                  # 5-fold cross-validation (standard)

CLASS_NAMES = ['Background', 'LV Cavity', 'Myocardium', 'Left Atrium']

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {pl.__version__}")
print(f"SMP: {smp.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"\nTraining config: {N_FOLDS}-fold CV, {EPOCHS} epochs × {BATCHES_PER_EPOCH} batches/epoch")
print(f"Total max iterations per fold: {EPOCHS * BATCHES_PER_EPOCH:,}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 2: CAMUSDataset with Albumentations augmentation
# ══════════════════════════════════════════════════════════════

def get_train_transforms():
    """Light affine-only augmentation (proven most impactful by nnU-Net ablation)."""
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Affine(
            scale=(0.9, 1.1),            # ±10%
            translate_percent=(-0.1, 0.1), # ±10%
            rotate=(-15, 15),             # ±15°
            border_mode=cv2.BORDER_CONSTANT,
            p=0.5,
        ),
    ])


class CAMUSDataset(Dataset):
    """
    CAMUS echocardiography dataset.
    samples = list of (image_path, mask_path) tuples.
    """

    def __init__(self, samples, img_size=256, augment=False):
        self.samples = samples
        self.img_size = img_size
        self.augment = augment
        self.transform = get_train_transforms() if augment else None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        # ── Load NIfTI ──
        img = nib.load(img_path).get_fdata().astype(np.float32)
        mask = nib.load(mask_path).get_fdata().astype(np.float32)

        # Handle 3D (take first slice)
        if img.ndim == 3:
            img = img[:, :, 0]
        if mask.ndim == 3:
            mask = mask[:, :, 0]

        # ── Resize ──
        img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        # ✅ GT rounding fix (keeps labels clean after resize)
        mask = np.round(mask).astype(np.int64)
        mask = np.clip(mask, 0, 3)

        # ── Normalize image to [0, 1] ──
        img_min, img_max = img.min(), img.max()
        if img_max > img_min:
            img = (img - img_min) / (img_max - img_min)
        else:
            img = np.zeros_like(img)

        # ── Augmentation ──
        if self.augment and self.transform is not None:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']

        # ── To tensor ──
        img = torch.FloatTensor(img).unsqueeze(0)       # (1, H, W)
        mask = torch.LongTensor(mask.copy())             # (H, W)

        return img, mask

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 3: Data Discovery + Fold Splitting + DataModule
# ══════════════════════════════════════════════════════════════

def discover_patient_samples(data_dir):
    """Scan dataset and return {patient_id: [(img_path, mask_path), ...]}."""
    patient_dirs = sorted(glob.glob(os.path.join(data_dir, 'patient*')))
    print(f"Total patient folders found: {len(patient_dirs)}")

    patient_samples = {}
    for pdir in patient_dirs:
        patient_id = os.path.basename(pdir)
        samples = []
        gt_files = sorted(glob.glob(os.path.join(pdir, '*_gt.nii.gz')))
        for gt_path in gt_files:
            img_path = gt_path.replace('_gt.nii.gz', '.nii.gz')
            if os.path.exists(img_path):
                samples.append((img_path, gt_path))
        if samples:
            patient_samples[patient_id] = samples

    total = sum(len(s) for s in patient_samples.values())
    print(f"Total samples: {total} across {len(patient_samples)} patients "
          f"({total / len(patient_samples):.1f} samples/patient)")
    return patient_samples


def get_fold_splits(patient_samples, n_folds=5, seed=42):
    """
    Patient-level k-fold split.
    Returns list of (train_sample_list, val_sample_list) per fold.
    """
    patient_ids = sorted(patient_samples.keys())
    rng = np.random.RandomState(seed)
    rng.shuffle(patient_ids)

    fold_size = len(patient_ids) // n_folds
    folds = []

    for fold_idx in range(n_folds):
        val_start = fold_idx * fold_size
        # Last fold absorbs remainder patients
        val_end = val_start + fold_size if fold_idx < n_folds - 1 else len(patient_ids)

        val_pids = patient_ids[val_start:val_end]
        train_pids = patient_ids[:val_start] + patient_ids[val_end:]

        train_samples = [s for pid in train_pids for s in patient_samples[pid]]
        val_samples = [s for pid in val_pids for s in patient_samples[pid]]

        folds.append((train_samples, val_samples))
        print(f"  Fold {fold_idx}: {len(train_pids)} train patients ({len(train_samples)} samples) | "
              f"{len(val_pids)} val patients ({len(val_samples)} samples)")

    return folds


class CAMUSDataModule(pl.LightningDataModule):
    """Simplified DataModule — takes pre-split sample lists, no internal splitting."""

    def __init__(self, train_samples, val_samples, batch_size=4,
                 img_size=256, augment=False, num_workers=4):
        super().__init__()
        self.train_samples = train_samples
        self.val_samples = val_samples
        self.batch_size = batch_size
        self.img_size = img_size
        self.augment = augment
        self.num_workers = num_workers

    def setup(self, stage=None):
        self.train_dataset = CAMUSDataset(
            self.train_samples, self.img_size, augment=self.augment
        )
        self.val_dataset = CAMUSDataset(
            self.val_samples, self.img_size, augment=False
        )

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size,
                          shuffle=True, num_workers=self.num_workers,
                          persistent_workers=True, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size,
                          num_workers=self.num_workers,
                          persistent_workers=True, pin_memory=True)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 4: SMP UNet + Pretrained ResNet34 + Deep Supervision
# ══════════════════════════════════════════════════════════════

class DeepSupSMPUNet(nn.Module):
    """
    SMP UNet with:
    - Pretrained ResNet34 encoder (ImageNet)
    - SCSE decoder attention (channel + spatial squeeze-excitation)
    - Deep supervision via auxiliary heads on intermediate decoder outputs

    Training:  returns (main_output, [aux_1, aux_2, aux_3, aux_4])
    Inference: returns main_output only
    """

    def __init__(self, encoder_name='resnet34', encoder_weights='imagenet',
                 in_channels=1, num_classes=4):
        super().__init__()

        self.base = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=in_channels,
            classes=num_classes,
            decoder_attention_type='scse',
            decoder_channels=(256, 128, 64, 32, 16),
        )

        # ── Deep supervision: hook intermediate decoder blocks ──
        self._intermediate = {}
        for i, block in enumerate(self.base.decoder.blocks[:-1]):  # blocks 0,1,2,3
            block.register_forward_hook(self._make_hook(i))

        # Auxiliary heads (1×1 conv → num_classes)
        ds_channels = [256, 128, 64, 32]
        self.aux_heads = nn.ModuleList([
            nn.Conv2d(ch, num_classes, kernel_size=1) for ch in ds_channels
        ])

        # Init aux heads with Kaiming
        for head in self.aux_heads:
            nn.init.kaiming_normal_(head.weight, mode='fan_out', nonlinearity='relu')
            nn.init.zeros_(head.bias)

    def _make_hook(self, idx):
        def hook(module, input, output):
            self._intermediate[idx] = output
        return hook

    def forward(self, x):
        H, W = x.shape[2:]
        main_out = self.base(x)

        if self.training:
            aux_outputs = []
            for i, head in enumerate(self.aux_heads):
                feat = self._intermediate[i]
                aux = head(feat)
                aux = F.interpolate(aux, size=(H, W), mode='bilinear', align_corners=False)
                aux_outputs.append(aux)
            return main_out, aux_outputs

        return main_out


# ── Sanity check ──
_m = DeepSupSMPUNet('resnet34', 'imagenet', in_channels=1, num_classes=NUM_CLASSES)
_m.train()
_x = torch.randn(2, 1, IMG_SIZE, IMG_SIZE)
_main, _aux = _m(_x)
print(f"✅ Train mode — Main: {_main.shape}, Aux: {[a.shape for a in _aux]}")

_m.eval()
_out = _m(_x)
print(f"✅ Eval mode  — Output: {_out.shape}")

print(f"✅ Total params: {sum(p.numel() for p in _m.parameters()) / 1e6:.1f}M")
del _m, _x, _main, _aux, _out

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 5: DiceCE Loss + Deep Supervision Wrapper
# ── UNCHANGED ──
# ══════════════════════════════════════════════════════════════

class DiceCELoss(nn.Module):
    """DiceCE 30/70 — proven best loss across all experiments."""

    def __init__(self, num_classes=4, ce_weight=0.3, dice_weight=0.7,
                 class_weights=None, smooth=1e-5):
        super().__init__()
        self.num_classes = num_classes
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth

        weight = class_weights if class_weights is not None else None
        self.ce = nn.CrossEntropyLoss(weight=weight)

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)

        probs = F.softmax(logits, dim=1)
        targets_oh = F.one_hot(targets, self.num_classes).permute(0, 3, 1, 2).float()

        dims = (0, 2, 3)
        inter = (probs * targets_oh).sum(dims)
        union = probs.sum(dims) + targets_oh.sum(dims)
        dice = (2.0 * inter + self.smooth) / (union + self.smooth)
        dice_loss = 1.0 - dice.mean()

        return self.ce_weight * ce_loss + self.dice_weight * dice_loss


class DeepSupLoss(nn.Module):
    """Wraps any base loss to add deep supervision from auxiliary outputs."""

    def __init__(self, base_criterion, aux_weights=(0.1, 0.1, 0.1, 0.1)):
        super().__init__()
        self.base_criterion = base_criterion
        self.aux_weights = aux_weights

    def forward(self, outputs, target):
        if isinstance(outputs, tuple):
            main_out, aux_outputs = outputs
            loss = self.base_criterion(main_out, target)
            for i, aux in enumerate(aux_outputs):
                if i < len(self.aux_weights):
                    loss = loss + self.aux_weights[i] * self.base_criterion(aux, target)
            return loss
        return self.base_criterion(outputs, target)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 6: Lightning Module — Differential LR + Per-class Dice
# ══════════════════════════════════════════════════════════════

class CardiacSegModule(pl.LightningModule):
    def __init__(self, model, criterion, lr=3e-4, encoder_lr_factor=0.1,
                 weight_decay=1e-4, epochs=100, num_classes=4):
        super().__init__()
        self.model = model
        self.criterion = criterion
        self.lr = lr
        self.encoder_lr_factor = encoder_lr_factor
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.num_classes = num_classes

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        outputs = self.model(x)
        loss = self.criterion(outputs, y)

        main_out = outputs[0] if isinstance(outputs, tuple) else outputs
        preds = main_out.argmax(dim=1)
        dice = self._mean_fg_dice(preds, y)

        self.log('train_loss', loss, prog_bar=True)
        self.log('train_dice', dice, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        outputs = self.model(x)
        loss = self.criterion(outputs, y)
        preds = outputs.argmax(dim=1)

        dice = self._mean_fg_dice(preds, y)
        class_dices = self._per_class_dice(preds, y)

        self.log('val_loss', loss, prog_bar=True)
        self.log('val_dice', dice, prog_bar=True)
        for c in range(1, self.num_classes):
            self.log(f'val_dice_c{c}', class_dices[c], prog_bar=False)
        return loss

    def _mean_fg_dice(self, preds, targets):
        dice_sum, count = 0.0, 0
        for c in range(1, self.num_classes):
            pred_c = (preds == c).float()
            tgt_c = (targets == c).float()
            inter = (pred_c * tgt_c).sum()
            union = pred_c.sum() + tgt_c.sum()
            if union > 0:
                dice_sum += 2.0 * inter / (union + 1e-8)
                count += 1
        return dice_sum / max(count, 1)

    def _per_class_dice(self, preds, targets):
        dices = {}
        for c in range(self.num_classes):
            pred_c = (preds == c).float()
            tgt_c = (targets == c).float()
            inter = (pred_c * tgt_c).sum()
            union = pred_c.sum() + tgt_c.sum()
            dices[c] = (2.0 * inter / (union + 1e-8)) if union > 0 else torch.tensor(1.0)
        return dices

    def configure_optimizers(self):
        encoder_params = list(self.model.base.encoder.parameters())
        decoder_params = (
            list(self.model.base.decoder.parameters()) +
            list(self.model.base.segmentation_head.parameters()) +
            list(self.model.aux_heads.parameters())
        )

        optimizer = torch.optim.AdamW([
            {'params': encoder_params, 'lr': self.lr * self.encoder_lr_factor},
            {'params': decoder_params, 'lr': self.lr},
        ], weight_decay=self.weight_decay)

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.epochs, eta_min=1e-6
        )
        return [optimizer], [scheduler]

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 7: 5-Fold Cross-Validation — nnU-Net style
# ══════════════════════════════════════════════════════════════

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. Discover dataset & prepare folds
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

patient_samples = discover_patient_samples(DATA_DIR)
print()
folds = get_fold_splits(patient_samples, n_folds=N_FOLDS, seed=SEED)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2. Train each fold
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

fold_results = {}

for fold_idx in range (N_FOLDS):
    print(f"\n{'═' * 60}")
    print(f"  FOLD {fold_idx + 1} / {N_FOLDS}")
    print(f"{'═' * 60}\n")

    train_samples, val_samples = folds[fold_idx]

    # ── DataModule ──
    dm = CAMUSDataModule(
        train_samples=train_samples,
        val_samples=val_samples,
        batch_size=BATCH_SIZE,
        img_size=IMG_SIZE,
        augment=True,
        num_workers=NUM_WORKERS,
    )
    dm.setup()
    print(f"  Train: {len(dm.train_dataset)} samples "
          f"({len(dm.train_dataset) // BATCH_SIZE} batches/full pass)")
    print(f"  Val:   {len(dm.val_dataset)} samples")

    # ── Fresh model (pretrained encoder reloaded each fold) ──
    model = DeepSupSMPUNet(
        encoder_name='resnet34',
        encoder_weights='imagenet',
        in_channels=1,
        num_classes=NUM_CLASSES,
    )

    # ── Loss ──
    base_criterion = DiceCELoss(
        num_classes=NUM_CLASSES,
        ce_weight=0.3,
        dice_weight=0.7,
        class_weights=None,
    )
    criterion = DeepSupLoss(base_criterion, aux_weights=(0.1, 0.1, 0.1, 0.1))

    # ── Lightning module ──
    module = CardiacSegModule(
        model, criterion,
        lr=LR,
        encoder_lr_factor=0.1,
        weight_decay=1e-4,
        epochs=EPOCHS,          # CosineAnnealing T_max = 1000
        num_classes=NUM_CLASSES,
    )

    # ── Callbacks ──
    fold_save_dir = os.path.join(SAVE_DIR, f'v3cv_fold{fold_idx}')
    os.makedirs(fold_save_dir, exist_ok=True)

    checkpoint_cb = ModelCheckpoint(
        dirpath=fold_save_dir,
        filename=f'v3cv-fold{fold_idx}-{{epoch}}-{{val_dice:.4f}}',
        monitor='val_dice',
        mode='max',
        save_top_k=1,
        verbose=True,
    )

    early_stop_cb = EarlyStopping(
        monitor='val_dice',
        patience=PATIENCE,
        mode='max',
        verbose=True,
    )

    # ── Trainer (nnU-Net style: 250 batches = 1 epoch) ──
    trainer = pl.Trainer(
        max_epochs=EPOCHS,
        accelerator='gpu',
        devices=1,
        callbacks=[checkpoint_cb, early_stop_cb],
        log_every_n_steps=10,
        limit_train_batches=BATCHES_PER_EPOCH,   # ← KEY: 250 batches per epoch
    )

    trainer.fit(module, dm)

    # ── Collect results ──
    best_dice = float(checkpoint_cb.best_model_score)
    fold_results[fold_idx] = {
        'best_dice': best_dice,
        'best_path': checkpoint_cb.best_model_path,
        'stopped_epoch': trainer.current_epoch,
    }

    print(f"\n✅ Fold {fold_idx} done — Best val Dice: {best_dice:.4f} "
          f"(stopped @ epoch {trainer.current_epoch})")

    # ── Clean up GPU between folds ──
    del model, module, trainer, dm, criterion, base_criterion
    torch.cuda.empty_cache()
    import gc; gc.collect()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3. Final report
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print(f"\n{'═' * 60}")
print("  5-FOLD CROSS-VALIDATION RESULTS (v3 architecture)")
print(f"{'═' * 60}")

dices = [fold_results[i]['best_dice'] for i in range(N_FOLDS)]
for i in range(N_FOLDS):
    r = fold_results[i]
    print(f"  Fold {i}: val_dice = {r['best_dice']:.4f}  "
          f"(stopped @ epoch {r['stopped_epoch']})")

mean_dice = np.mean(dices)
std_dice = np.std(dices)

print(f"\n  {'─' * 40}")
print(f"  Mean ± Std:  {mean_dice:.4f} ± {std_dice:.4f}")
print(f"  Min / Max:   {np.min(dices):.4f} / {np.max(dices):.4f}")
print(f"  {'─' * 40}")

print(f"\n  Comparison (mean foreground Dice):")
print(f"  ┌─────────────────────────────────────────────────┐")
print(f"  │ v3 single split (val, biased):     0.9140       │")
print(f"  │ v3 5-fold CV (this run):           {mean_dice:.4f} ± {std_dice:.4f} │")
print(f"  │ nnU-Net literature (CAMUS):        ~0.893       │")
print(f"  │ Lightweight U-Net (CAMUS):         ~0.890       │")
print(f"  │ CAMUS-HeartNet ensemble:           ~0.911       │")
print(f"  └─────────────────────────────────────────────────┘")
print(f"{'═' * 60}")

# Save fold results for later use in evaluation
import json
results_path = os.path.join(SAVE_DIR, 'v3cv_fold_results.json')
with open(results_path, 'w') as f:
    json.dump({
        'fold_results': {str(k): v for k, v in fold_results.items()},
        'mean_dice': float(mean_dice),
        'std_dice': float(std_dice),
        'config': {
            'epochs': EPOCHS,
            'batches_per_epoch': BATCHES_PER_EPOCH,
            'patience': PATIENCE,
            'n_folds': N_FOLDS,
            'batch_size': BATCH_SIZE,
            'lr': LR,
            'encoder': 'resnet34',
            'img_size': IMG_SIZE,
        }
    }, f, indent=2)
print(f"\nResults saved to {results_path}")